# 1. Forecast detectives
## Can AI Predict the Future?
**45 minutes | pairs welcome | 4 milestones**

Today we test a promise, not prove that AI can know the future.
By the end you can prepare a time series, make a leak-free split,
create a seasonal baseline, and interpret a pretrained-model forecast.

**RUN THIS** = supplied code. **YOUR TURN** = replace `None` / write code.
Hints and solutions are below each exercise. Getting help counts as progress.
There are no points for speed.

### Before touching the keyboard (2 minutes)
Predict who will win: repeat last week, a classical statistical model, or
a pretrained transformer. Write **one reason**. Keep it for the exit vote.

## RUN THIS — get the workshop files
**Colab:** upload the trusted workshop ZIP when asked (the one you get
from the repository's green **Code → Download ZIP** button, or from the
organizer). It contains one folder with the data and helpers. Each
notebook may have its own runtime. **Local:** start Jupyter from the
workshop folder.
This cell does not install anything or download model weights.

In [ ]:
from pathlib import Path
import os
import sys
import zipfile

here = Path.cwd()
found = next(here.glob("*/workshop.py"), None)  # an already extracted ZIP
if (here / "workshop.py").exists():
    ROOT = here
elif (here.parent / "workshop.py").exists():
    ROOT = here.parent
elif found:
    ROOT = found.parent
else:
    if "google.colab" not in sys.modules and not os.environ.get("COLAB_RELEASE_TAG"):
        raise FileNotFoundError("Extract the ZIP and start Jupyter in the workshop folder.")
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one workshop ZIP.")
    archive_path = Path(next(iter(uploaded)))
    with zipfile.ZipFile(archive_path) as archive:
        tops = {name.split("/")[0] for name in archive.namelist() if name.strip("/")}
        if len(tops) != 1:
            raise ValueError("The ZIP must contain exactly one top-level folder.")
        top = tops.pop()
        if top in ("", ".", "..") or Path(top).is_absolute():
            raise ValueError("Unsafe folder name in the ZIP.")
        for member in archive.infolist():
            destination = (here / member.filename).resolve()
            if not destination.is_relative_to((here / top).resolve()):
                raise ValueError("Unsafe path in the ZIP.")
        archive.extractall(here)
    ROOT = here / top
    if not (ROOT / "workshop.py").exists():
        raise ValueError("workshop.py not found: is this the workshop ZIP?")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Workshop files:", ROOT)

### SETUP ONLY — do this before the workshop
Set `INSTALL_CORE = True` if the core packages are missing, and
`INSTALL_FOUNDATION = True` to install the foundation-model packages.
You need the foundation packages to run the pretrained models yourself;
they are much larger than the core (see `SETUP.md`). Without them you
can still use the instructor's pre-computed forecasts. After
installation, restart the session if requested; rerun with both flags
`False`. A GPU is not necessary.
Read `SETUP.md` for downloads, CPU fallback, and troubleshooting.

In [ ]:
import subprocess

def run(command):
    # Run a command and show its output live, like a terminal.
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    return process.wait()

INSTALL_CORE = False
INSTALL_FOUNDATION = False
for wanted, name in ((INSTALL_CORE, "core"), (INSTALL_FOUNDATION, "foundation")):
    if wanted:
        print(f"Installing {name} packages: this can take several minutes...")
        if run([sys.executable, "-m", "pip", "install", "-r", f"requirements/{name}.txt"]):
            raise RuntimeError(f"Installing {name} failed. Read the messages above.")
        print(f"Finished installing {name} packages.")

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML
from workshop import (
    load_series, demo_split, make_model, validate_prediction,
    benchmark_series, splitter, run_benchmark, leaderboard, load_run,
    CONTEXT, HORIZON, N_FOLDS, SEASON, BASELINES, FOUNDATIONS,
)
if run([sys.executable, "scripts/check_environment.py"]):
    raise RuntimeError("The environment check failed. Read the messages above.")

# Plot helpers: one colour per entity, colour-blind-safe (Okabe-Ito).
pio.templates.default = "plotly_white"
BLUE, ORANGE, GREEN, INK = "#0072B2", "#D55E00", "#009E73", "#333333"
MODEL_COLORS = {
    "seasonal-naive": ORANGE, "arima": "#E69F00", "ets": "#CC79A7",
    "chronos2": GREEN, "timesfm25": "#56B4E9", "t0-gamma": BLUE,
}

def new_figure(title, ylabel="Births per day"):
    fig = go.Figure()
    fig.update_layout(
        title=title, yaxis_title=ylabel, hovermode="x unified", height=380,
        legend=dict(orientation="h", y=-0.2), margin=dict(t=60, b=40),
    )
    return fig

def add_line(fig, series, name, color, dash="solid", width=2, markers=False):
    fig.add_scatter(
        x=series.index, y=series.to_numpy(), name=name,
        mode="lines+markers" if markers else "lines",
        line=dict(color=color, dash=dash, width=width),
    )

def mark_cutoff(fig, cutoff):
    fig.add_shape(
        type="line", x0=cutoff, x1=cutoff, y0=0, y1=1, yref="paper",
        line=dict(color=ORANGE, dash="dot"),
    )
    fig.add_annotation(
        x=cutoff, y=1, yref="paper", text="Forecast cutoff", showarrow=False,
        xanchor="right", yanchor="top", font=dict(color="#666666"),
    )

y = load_series()
milestones = {}

def celebrate(key, message):
    milestones[key] = message
    display(HTML(
        f"<progress value='{len(milestones)}' max='4'></progress> "
        f"<b>{len(milestones)}/4 milestones:</b> {message}"
    ))
    print("Completed:", ", ".join(milestones))

> **Curious what a helper does?** Functions like `demo_split` or
> `leaderboard` come from `workshop.py`. In any cell, run
> `help(demo_split)` (or `demo_split?` in Jupyter) to read what it does.
> You never have to edit that file.

## 1. Data detective (7 minutes)
We have **420 daily aggregate U.S. birth counts**, from FiveThirtyEight/SSA,
licensed CC BY 4.0. See `data/README.md` for attribution and the pinned source.
This is not a medical decision tool.

Frequency is **daily**; the seasonal period we will test is **7 days**.
Frequency and seasonality are different concepts.

We reserve the **last 42 days** for notebook 2's evaluation before looking
at them. No smoothing or imputation uses the future. `sktime` accepts this
regular pandas Series directly—datasets need not be built into sktime.

In [ ]:
development = y.iloc[:-N_FOLDS * HORIZON]
print("Development range:", development.index.min(), "to", development.index.max())
print("Daily frequency:", development.index.freqstr)
display(development.head().to_frame())

### YOUR TURN — Plot the series yourself (2 minutes)

Draw the development data as a line chart with `px.line(...)`. Give it a title, then look: what repeats? What might surprise a model? Hover, zoom, and drag over the chart to explore.

In [ ]:
fig = None  # Replace None with px.line(development, title="...")
assert fig is not None, "Hint: px.line(development, title=...)"
assert len(fig.data) == 1 and len(fig.data[0].y) == len(development), "Plot the whole development series once."
fig.update_layout(xaxis_title="Date", yaxis_title="Births per day")
fig.show()

**Hint:** `px.line` accepts a pandas Series directly: the index becomes the x axis.

<details><summary>Show solution (copy into the exercise cell when you are ready)</summary>

```python
fig = px.line(development, title="What repeats? What might surprise a model?")
fig.update_layout(xaxis_title="Date", yaxis_title="Births per day")
fig.show()
```

</details>

In [ ]:
weekday_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
frame = development.to_frame("births")
frame["weekday"] = frame.index.dayofweek
frame["week_start"] = frame.index - pd.to_timedelta(frame["weekday"].to_numpy(), unit="D")
by_week = frame.pivot(index="weekday", columns="week_start", values="births").dropna(axis=1)
fig = new_figure("Weeks look alike: one thin line per week (last 12), development data only")
for i, week in enumerate(by_week.columns[-12:]):
    fig.add_scatter(
        x=weekday_names, y=by_week[week], mode="lines", hoverinfo="skip",
        name="Individual weeks", showlegend=(i == 0),
        line=dict(color=BLUE, width=1), opacity=0.45,
    )
fig.add_scatter(
    x=weekday_names, y=by_week.mean(axis=1), name="Average week",
    line=dict(color=INK, width=3),
)
fig.update_layout(xaxis_title="Day of week")
fig.show()

### YOUR TURN — Spot the weekly pattern (3 minutes)

Calculate the mean count for each weekday using **development only**. Monday is 0. Discuss one possible explanation without treating it as proven causation.

In [ ]:
weekday_means = None  # Group development by development.index.dayofweek, then mean
assert weekday_means is not None, "Hint: development.groupby(...).mean()"
assert len(weekday_means) == 7, "We need one average per weekday."
fig = px.bar(
    x=weekday_means.index, y=weekday_means.to_numpy(),
    title="Development data only: mean by weekday",
    labels={"x": "Weekday: Monday=0", "y": "Mean births per day"},
)
fig.show()
celebrate("detective", "You found structure worth forecasting.")

**Hint:** Dates have `.dayofweek`; grouping is descriptive, not a forecast.

<details><summary>Show solution (copy into the exercise cell when you are ready)</summary>

```python
weekday_means = development.groupby(development.index.dayofweek).mean()
assert len(weekday_means) == 7
fig = px.bar(
    x=weekday_means.index, y=weekday_means.to_numpy(),
    title="Development data only: mean by weekday",
    labels={"x": "Weekday: Monday=0", "y": "Mean births per day"},
)
fig.show()
celebrate("detective", "You found structure worth forecasting.")
```

</details>

## 2. A time machine with rules (10 minutes)
A **forecast origin/cutoff** is the last date we are allowed to observe.
**Context** is the historical data supplied to the model. A **horizon**
specifies how far ahead to predict, here days 1 through 14.

We use the last 14 **development** days as a demonstration holdout and
the preceding 168 days as context. Notebook 2 has a separate evaluation
period. Never use a random train/test split for this forecasting task.

### YOUR TURN — Build a leak-free demo split (4 minutes)

Select the demo test window, the immediately preceding 168 training days, and relative horizon 1–14. Do not include any test value in training.

In [ ]:
y_train = None
y_test = None
fh = None
assert y_train is not None and y_test is not None and fh is not None, "Fill all three."
expected_train, expected_test = demo_split(y)
pd.testing.assert_series_equal(y_train, expected_train)
pd.testing.assert_series_equal(y_test, expected_test)
assert list(fh) == list(range(1, 15)), "Forecast horizons start at 1, not 0."
assert y_train.index.max() < y_test.index.min()
celebrate("cutoff", "The future stayed out of training.")

**Hint:** `iloc[-14:]` selects the final 14 observations; the training slice stops before them.

<details><summary>Show solution (copy into the exercise cell when you are ready)</summary>

```python
y_train = development.iloc[-CONTEXT-HORIZON:-HORIZON]
y_test = development.iloc[-HORIZON:]
fh = np.arange(1, HORIZON + 1)
expected_train, expected_test = demo_split(y)
pd.testing.assert_series_equal(y_train, expected_train)
pd.testing.assert_series_equal(y_test, expected_test)
assert list(fh) == list(range(1, 15))
assert y_train.index.max() < y_test.index.min()
celebrate("cutoff", "The future stayed out of training.")
```

</details>

In [ ]:
fig = new_figure("Ask: could I have known this value at the cutoff?")
add_line(fig, y_train.iloc[-42:], "Available history", BLUE)
add_line(fig, y_test, "Hidden future (revealed for scoring)", INK, dash="dash")
mark_cutoff(fig, y_train.index[-1])
fig.show()

## 3. The baseline is a contestant, not a straw person (10 minutes)
**Seasonal naive:** repeat the last observed week. For horizons beyond
seven days, repeat that same available week—not actual future observations.
It has no learned neural weights, but often sets a high bar.

`fit(y_train, fh=fh)` remembers the data and forecasting horizon.
`predict()` returns timestamp-aligned forecasts.
**MAE** = average absolute difference between actual and prediction;
its unit here is births/day. Lower is better.

### YOUR TURN — Forecast with seasonal naive (4 minutes)

Create a `NaiveForecaster` with `strategy='last'` and the right seasonal period; fit it on training only, then predict.

In [ ]:
from sktime.forecasting.naive import NaiveForecaster
naive = None
assert naive is not None, "Create NaiveForecaster(strategy='last', sp=...)."
# YOUR TURN: call fit, then replace None with predict()
naive_pred = None
assert naive_pred is not None, "Fit on y_train, then call predict()."
validate_prediction(naive_pred, y_test)
np.testing.assert_allclose(naive_pred, np.tile(y_train.iloc[-7:], 2))
celebrate("baseline", "You made a real forecast with a sensible baseline.")

**Hint:** Seasonality is 7, not horizon 14. The same fit/predict interface is used for the AI models.

<details><summary>Show solution (copy into the exercise cell when you are ready)</summary>

```python
from sktime.forecasting.naive import NaiveForecaster
naive = NaiveForecaster(strategy="last", sp=7)
naive.fit(y_train, fh=fh)
naive_pred = naive.predict()
validate_prediction(naive_pred, y_test)
np.testing.assert_allclose(naive_pred, np.tile(y_train.iloc[-7:], 2))
celebrate("baseline", "You made a real forecast with a sensible baseline.")
```

</details>

In [ ]:
from sktime.performance_metrics.forecasting import mean_absolute_error
print(f"Demo seasonal-naive MAE: {mean_absolute_error(y_test, naive_pred):,.1f} births/day")
fig = new_figure("Baseline forecast: does the shape make sense?")
add_line(fig, y_train.iloc[-28:], "History", BLUE)
add_line(fig, y_test, "Actual", INK, width=3)
add_line(fig, naive_pred, "Seasonal naive", ORANGE, dash="dash", markers=True)
mark_cutoff(fig, y_train.index[-1])
fig.show()

## 4. A pretrained model joins (13 minutes)
Foundation models learn reusable patterns from many series. Here
**zero-shot** means no weight training on our workshop series. `fit` still
supplies observed history and can load a large checkpoint.

- **Chronos-2:** encoder-only patched transformer; time/group attention.
- **TimesFM 2.5:** decoder-only patched transformer.
- **t0-gamma:** decoder-style patch transformer with time/covariate attention.

This is a univariate task without exogenous variables. It does **not**
exercise everything these models can do. See `TUTORIAL.md` for the important
differences between native models and sktime wrappers.

**Choose one**, not all three in this runtime. Default is saved-results
mode. If its folder is absent, the code says so rather than inventing an
AI curve. The instructor must supply real model results from rehearsal.
On a prepared GPU runtime, set `LIVE = True`.

Running t0-gamma means you accept its license terms (linked in `workshop.py`).
Its public gamma checkpoint is selected explicitly; the adapter's default
alpha is not gamma. Do not use `model_path=None`—that is an untrained model.

In [ ]:
CHOSEN_MODEL = "chronos2"  # Or "timesfm25" or "t0-gamma"
LIVE = False
DEVICE = "cpu"  # CPU is fine for this small series; use "cuda" on a GPU runtime
foundation_pred = None
evidence_label = None

if CHOSEN_MODEL not in FOUNDATIONS:
    raise ValueError("Choose chronos2, timesfm25, or t0-gamma.")
if LIVE:
    foundation = make_model(CHOSEN_MODEL, DEVICE)
    foundation.fit(y_train, fh=fh)
    foundation_pred = foundation.predict()
    evidence_label = "live inference"
else:
    # Prefer a run you made yourself (artifacts-live), else the instructor's.
    saved = ROOT / "artifacts-live" / CHOSEN_MODEL
    source = "your own run"
    if not saved.exists():
        saved = ROOT / "artifacts" / CHOSEN_MODEL
        source = "instructor run"
    if saved.exists():
        _, _, metadata = load_run(saved, y)
        if metadata["model"] != CHOSEN_MODEL:
            raise ValueError("Saved model does not match the selected model.")
        demo = pd.read_csv(saved / "demo.csv", parse_dates=["date"]).set_index("date")
        np.testing.assert_array_equal(demo["actual"], y_test)
        foundation_pred = demo["prediction"].asfreq("D")
        evidence_label = f"saved real inference ({source}); not run in this notebook"
        print("Provenance:", metadata)
    else:
        print(f"NOT EVALUATED: no saved {CHOSEN_MODEL} run. Use prepared live mode")
        print("or ask the instructor for rehearsal artifacts. No AI forecast is shown.")
if foundation_pred is not None:
    validate_prediction(foundation_pred, y_test)
    fig = new_figure(f"{CHOSEN_MODEL}: {evidence_label}")
    add_line(fig, y_train.iloc[-28:], "History", BLUE)
    add_line(fig, y_test, "Actual", INK, width=3)
    add_line(fig, naive_pred, "Seasonal naive", ORANGE, dash="dash")
    add_line(fig, foundation_pred, CHOSEN_MODEL, GREEN, dash="dashdot", markers=True)
    mark_cutoff(fig, y_train.index[-1])
    fig.show()
    display(pd.Series({
        "seasonal-naive": mean_absolute_error(y_test, naive_pred),
        CHOSEN_MODEL: mean_absolute_error(y_test, foundation_pred),
    }, name="Demo MAE (births/day)").to_frame())

### YOUR TURN — explain, don't just rank (2 minutes)
Write two sentences below. If an AI run is unavailable, explain what
evidence would be needed; do not claim it won or lost.

In [ ]:
reflection = ""  # Where did the forecast work/fail? Why is one split not enough?
if reflection.strip():
    celebrate("explorer", "You distinguished a claim from the evidence behind it.")
else:
    print("Write two sentences, then rerun. You can also share them verbally with a partner.")

## Checkpoint: 45 minutes
You can now forecast, plot, and ask whether the result is fair.
Save this notebook and open **02_fair_forecast_arena.ipynb**.
A new Colab notebook may need the ZIP/setup again. It does not need
variables from this notebook.

**Optional discussion:** a quantile interval is not a guarantee. The
native Chronos-2 model produces quantiles, but this pinned sktime wrapper
does not expose prediction intervals. We will not fabricate uncertainty
bands from point predictions.